In [15]:
import numpy as np
import pandas as pd

from pathlib import Path
from huggingface_hub import login
from huggingface_hub import hf_hub_download
from sklearn.model_selection import train_test_split

In [5]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Nexar_Crash_Prediction"
)

DATASET_DIR = (
    PROJECT_ROOT / "dataset" / "Nexar_DashCam_Dataset"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODELS_DIR = PROJECT_ROOT / "models"

for directory in [
    PROJECT_ROOT,
    OUTPUT_DIR,
    MODELS_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Project root:", PROJECT_ROOT)
print("Dataset directory:", DATASET_DIR)
print("Dataset exists:", DATASET_DIR.exists())

Project root: /content/drive/MyDrive/Nexar_Crash_Prediction
Dataset directory: /content/drive/MyDrive/Nexar_Crash_Prediction/dataset/Nexar_DashCam_Dataset
Dataset exists: False


In [8]:
login()

In [11]:
REPO_ID = "nexar-ai/nexar_collision_prediction"

positive_metadata_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="train/positive/metadata.csv",
    repo_type="dataset"
)

negative_metadata_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="train/negative/metadata.csv",
    repo_type="dataset"
)

print("Metadata files downloaded successfully!")

metadata.csv:   0%|          | 0.00/33.3k [00:00<?, ?B/s]

metadata.csv:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Metadata files downloaded successfully!


In [14]:
REPO_ID = "nexar-ai/nexar_collision_prediction"

if "positive_df" not in globals():
    positive_path = hf_hub_download(
        repo_id=REPO_ID,
        filename="train/positive/metadata.csv",
        repo_type="dataset"
    )
    positive_df = pd.read_csv(positive_path)

if "negative_df" not in globals():
    negative_path = hf_hub_download(
        repo_id=REPO_ID,
        filename="train/negative/metadata.csv",
        repo_type="dataset"
    )
    negative_df = pd.read_csv(negative_path)

print("Metadata is ready.")

Metadata is ready.


In [16]:
positive_data = positive_df.copy()
negative_data = negative_df.copy()

positive_data["label"] = 1
negative_data["label"] = 0

all_videos = pd.concat(
    [positive_data, negative_data],
    ignore_index=True
)

all_videos = all_videos.drop_duplicates(
    subset="file_name"
).reset_index(drop=True)

train_videos, val_videos = train_test_split(
    all_videos,
    test_size=0.2,
    random_state=42,
    stratify=all_videos["label"]
)

train_videos = train_videos.copy()
val_videos = val_videos.copy()

train_videos["split"] = "train"
val_videos["split"] = "validation"

video_split = pd.concat(
    [train_videos, val_videos],
    ignore_index=True
)

OUTPUT_DIR = Path("/content/nexar_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

video_split.to_csv(
    OUTPUT_DIR / "video_split.csv",
    index=False
)

print("Train videos:", len(train_videos))
print("Validation videos:", len(val_videos))

Train videos: 1200
Validation videos: 300


In [17]:
train_ids = set(train_videos["file_name"])
val_ids = set(val_videos["file_name"])

overlap = train_ids.intersection(val_ids)

print("Overlapping videos:", len(overlap))

print("\nClass distribution:")
display(
    video_split.groupby(
        ["split", "label"]
    ).size().unstack(fill_value=0)
)

Overlapping videos: 0

Class distribution:


label,0,1
split,,
train,600,600
validation,150,150
